___

# <font color= #99C8F5> **M-Nist GAN with FID Metric** </font>
#### <font color= #2E9AFE> `Deep Learning`</font>
<Strong> Sofía Maldonado, Óscar Josué Rocha & Viviana Toledo </Strong>

_26/03/2026._

___

In [146]:
# General
import numpy as np
import os
import sys

# Visualization
import matplotlib.pyplot as plt

# Modeling
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim as optim
import torch.utils.data
from torchvision import datasets, transforms
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision.datasets as dset
import torchvision.utils as vutils

# Import Inception
from torchvision import models
from scipy import linalg
import torch.nn.functional as F
from PIL import Image

# <font color= #99C8F5> **Hyperparameters and Data** </font>

In [138]:
CUDA = True
DATA_PATH = './data'
BATCH_SIZE = 256
IMAGE_CHANNEL = 1                   # 1 for grayscale, 3 for RGB
LATENT_DIM = 100                         # Latent Vector Dimension
G_HIDDEN = 192                    # Generator Neurons (features)
X_DIM = 64
D_HIDDEN = 64                       # Discriminator Neurons (features)
EPOCHS = 5
REAL_LABEL = 1
FAKE_LABEL = 0
LR = 2e-4
seed = 1

In [139]:
CUDA = CUDA and torch.cuda.is_available()
print("PyTorch version: {}".format(torch.__version__))
if CUDA:
    print("CUDA version: {}\n".format(torch.version.cuda))

if CUDA:
    torch.cuda.manual_seed(seed)
device = torch.device("cuda:0" if CUDA else "cpu")
cudnn.benchmark = True

PyTorch version: 2.10.0+cu130
CUDA version: 13.0



In [140]:
# Data preprocessing
dataset = dset.MNIST(root=DATA_PATH, download=True,
                     transform=transforms.Compose([
                     transforms.Resize(X_DIM),
                     transforms.ToTensor(),
                     transforms.Normalize((0.5,), (0.5,))
                     ]))

# Dataloader
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE,
                                         shuffle=True, num_workers=2)

In [148]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            # input layer
            nn.ConvTranspose2d(LATENT_DIM, G_HIDDEN * 8, 4, 1, 0, bias=False),               # Receives "noise" which is the latent space
            nn.BatchNorm2d(G_HIDDEN * 8),                                               # Normalization
            nn.ReLU(True),                                                              # Activation Function
            # 1st hidden layer
            nn.ConvTranspose2d(G_HIDDEN * 8, G_HIDDEN * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(G_HIDDEN * 4),
            nn.ReLU(True),
            # 2nd hidden layer
            nn.ConvTranspose2d(G_HIDDEN * 4, G_HIDDEN * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(G_HIDDEN * 2),
            nn.ReLU(True),
            # 3rd hidden layer
            nn.ConvTranspose2d(G_HIDDEN * 2, G_HIDDEN, 4, 2, 1, bias=False),
            nn.BatchNorm2d(G_HIDDEN),
            nn.ReLU(True),
            # output layer
            nn.ConvTranspose2d(G_HIDDEN, IMAGE_CHANNEL, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z_dim):
        return self.model(z_dim)


# Load the pre-trained generator
generator = Generator()
generator.load_state_dict(torch.load("mnist_gan_demo/generator.pth", map_location="cpu", weights_only=False))
generator.eval()


def generate_digit():
    """Generate a random MNIST digit using the GAN."""
    with torch.no_grad():
        z_dim = torch.randn(1, 100)
        z_dim = z_dim.view(1,100,1,1)
        fake_image = generator(z_dim)
        # Convert from [-1, 1] to [0, 255]
        img_array = ((fake_image.squeeze().numpy() + 1) / 2 * 255).astype(np.uint8)
        img = Image.fromarray(img_array, mode="L")
        # Resize for better visibility
        img = img.resize((280, 280), Image.NEAREST)
        return img_array
    

# <font color= #99C8F5> **Generating Images** </font>

In [170]:
img_gen=generate_digit()

img_real = dataset[0]

# <font color= #99C8F5> **Custom Difference** </font>

In [185]:
fake_mean = img_gen.mean()

real_mean = img_real[0].mean()

In [190]:
fake_diff = img_gen - fake_mean

real_diff = img_real[0] - real_mean

In [193]:
real_norm = linalg.norm(real_diff)

fake_norm = linalg.norm(fake_diff)

In [195]:
real_norm - fake_norm 

np.float64(-5803.3617494255805)